In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
from sklearn.preprocessing import OneHotEncoder

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

%matplotlib inline



# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: (Read the dataset Q1_data.csv using read_csv())


csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:Inspect the first few rows using head()
df.head()

In [ ]:
# Task 3: Write your code here: Display dataset information using info()

df.info()

In [ ]:
# Task 4: Write your code here:Show statistical description using describe()

df.describe()

In [ ]:
# Task 5: Write your code here:Plot the target distribution (delivery_time)
target_column="Delivery_Time"
df[target_column].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({target_column})")
plt.xlabel(target_column)
plt.ylabel("Frequency")
plt.grid(False)

plt.show()


In [ ]:
# Task 1: Write your code here:
df=df.drop(columns=["Order_ID"])


In [ ]:
# Task 2: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] >= 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
display(missing_data.head(10))

#not much data is missing from any column, best option is to fill with mean or mode
num_column=df.select_dtypes(exclude=["object"]).columns
cat_column=df.select_dtypes(include=["object"]).columns

for x in num_column:
  df[x]=df[x].fillna(df[x].mean())


for x in cat_column:
  df[x]=df[x].fillna(df[x].mode()[0])


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.drop_duplicates(inplace=True)
df.shape

In [ ]:

# Task 4: Write your code here:


onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
df=pd.get_dummies(df,cat_column, drop_first=True)

df

In [ ]:
# Task 5: Write your code here:


from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
df[num_column] = scaler.fit_transform(df[num_column])
df.head()


In [ ]:
# Task 6: Write your code here:

df[target_column].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({target_column})")
plt.xlabel(target_column)
plt.ylabel("Frequency")
plt.grid(False)

plt.show()

#data is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df.drop(target_column, axis=1).astype(float)
y = df[target_column].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200)
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mae': [],'mse': [], 'rmse': [], 'r2': []}

n_splits=5

kf = KFold(n_splits=5, shuffle=True, random_state=42)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    mae= mean_absolute_error(y_test,y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)
    all_results[model_name]["mae"].append(mae)

for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  MAE:  {np.mean(all_results[model_name]['mae']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:
models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000)
}

# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  y_pred=[]
  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict

    y_pred.append(model.predict(X_test))

    # Calculate metrics


  mae= mean_absolute_error(y_test,(y_pred[0]+y_pred[1])/2)
  print(mae)